# Jacobian analysis on held-out neurons

Reads the `interpret` zarrs for the **base pretrained** and the **LoRA fine-tuned** models on the held-out neuron cell type, and runs three analyses:

1. Neuron TF fingerprint (per-motif activator/repressor contribution).
2. Motif importance vs TF mRNA expression (sanity check using `NrMotifV1` cluster-to-TF mapping).
3. Base vs fine-tuned motif-importance diff (what fine-tuning learned).

Outputs land in `output/finetune_multiome1_human/jacobian_analysis/`.


In [ ]:
import json
import os
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import zarr
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_context("notebook")
sns.set_style("whitegrid")

COURSE_WORK = Path(os.environ.get("GET_COURSE_WORK", Path.home() / "GET_course_work"))
PREPROC_DIR = Path(os.environ.get("GET_PREPROCESSED_DIR", COURSE_WORK / "multiome_1" / "preprocessed"))
OUT_ROOT = Path(
    os.environ.get(
        "GET_MULTIOME_OUTPUT_ROOT",
        COURSE_WORK / "output" / "finetune_multiome1_human",
    )
)

BASE_ZARR = Path(os.environ.get("GET_BASE_INTERPRET_ZARR", OUT_ROOT / "interpret_base_neurons" / "neurons.zarr"))
FT_ZARR = Path(os.environ.get("GET_FT_INTERPRET_ZARR", OUT_ROOT / "interpret_ft_neurons" / "neurons.zarr"))
SRC_ZARR = Path(os.environ.get("GET_MULTIOME_ZARR", PREPROC_DIR / "multiome1_human.zarr"))
RNA_CSV = Path(os.environ.get("GET_HOLDOUT_RNA_CSV", PREPROC_DIR / "neurons.rna.csv"))
OUT_DIR = Path(os.environ.get("GET_JACOBIAN_OUT_DIR", OUT_ROOT / "jacobian_analysis"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output dir:", OUT_DIR)


## Gene x motif contribution matrices

We compute each sample's per-motif contribution as `mean_r ( jacobian[r, m] * input[r, m] )` (mirroring `OneGeneJacobian.motif_summary(stats="mean")` with `multiply_input=True`). For each sample we pick the jacobian whose output-strand (`jacobians/exp/{0,1}`) matches the gene's strand, then group by gene name and average across TSS. Streaming in chunks to keep memory under ~1 GB.

In [ ]:
src = zarr.open(str(SRC_ZARR), mode="r")
MOTIF_NAMES = np.array([m.strip() for m in src["motif_names"][:]])
FEATURES = np.concatenate([MOTIF_NAMES, np.array(["Accessibility"])])
print("n motifs:", len(MOTIF_NAMES), "n features (motif + accessibility):", len(FEATURES))
print("first 5 motifs:", MOTIF_NAMES[:5], "last 3:", MOTIF_NAMES[-3:])


def compute_gene_by_motif(zarr_path, chunk=1024):
    """
    Returns a DataFrame (n_unique_genes x 283) where:
      - columns = MOTIF_NAMES (282) + 'Accessibility' (1, kept for ATAC jacobian analyses but NOT used for motif-only outputs)
      - values  = mean over 200 regions of (jacobian * input) per sample,
                  then averaged across all TSS per gene.
    Uses the jacobian output slot matching each sample's strand.
    """
    z = zarr.open(str(zarr_path), mode="r")
    gene_names = np.array([g.strip() for g in z["available_genes"][:]])
    strands = z["strand"][:].astype(int)
    n = gene_names.shape[0]
    contrib = np.zeros((n, len(FEATURES)), dtype=np.float32)

    j0 = z["jacobians/exp/0/input/region_motif"]
    j1 = z["jacobians/exp/1/input/region_motif"]
    inp = z["input"]

    for start in range(0, n, chunk):
        end = min(start + chunk, n)
        s_chunk = strands[start:end]
        inp_chunk = inp[start:end].astype(np.float32)
        j0_chunk = j0[start:end]
        j1_chunk = j1[start:end]
        picked = np.where(s_chunk[:, None, None] == 0, j0_chunk, j1_chunk)
        contrib[start:end] = (picked * inp_chunk).mean(axis=1)

    df = pd.DataFrame(contrib, index=gene_names, columns=FEATURES)
    df = df.groupby(level=0, sort=True).mean()
    return df


print("Computing gene-by-motif for FT ...")
gbm_ft = compute_gene_by_motif(FT_ZARR)
print("  shape:", gbm_ft.shape)
print("Computing gene-by-motif for BASE ...")
gbm_base = compute_gene_by_motif(BASE_ZARR)
print("  shape:", gbm_base.shape)

assert gbm_ft.index.equals(gbm_base.index), "gene index mismatch between base and FT"

gbm_ft.to_csv(OUT_DIR / "gene_by_motif_ft.csv.gz", compression="gzip")
gbm_base.to_csv(OUT_DIR / "gene_by_motif_base.csv.gz", compression="gzip")
print("Saved gene-by-motif matrices.")

## Analysis 1: Neuron TF fingerprint

For each motif, aggregate the signed contribution `J*x` across all neuron genes and regions, split into activator- (positive) and repressor- (negative) sums. Produce top-motif tables + a summary figure.

In [ ]:
mot_ft = gbm_ft[MOTIF_NAMES]
mot_base = gbm_base[MOTIF_NAMES]

act_sum = mot_ft.clip(lower=0).sum(axis=0)
rep_sum = mot_ft.clip(upper=0).sum(axis=0)
abs_sum = mot_ft.abs().sum(axis=0)

fingerprint = pd.DataFrame({
    "motif": MOTIF_NAMES,
    "activator_sum": act_sum.values,
    "repressor_sum": rep_sum.values,
    "abs_sum": abs_sum.values,
})
fingerprint["abs_rank"] = fingerprint["abs_sum"].rank(ascending=False).astype(int)
fingerprint = fingerprint.sort_values("abs_rank").reset_index(drop=True)
fingerprint.to_csv(OUT_DIR / "fingerprint_neurons.csv", index=False)

top_act = fingerprint.sort_values("activator_sum", ascending=False).head(15)
top_rep = fingerprint.sort_values("repressor_sum", ascending=True).head(15)
top_abs = fingerprint.sort_values("abs_sum", ascending=False).head(25)

print("Top 15 activators:")
print(top_act[["motif", "activator_sum"]].to_string(index=False))
print("\nTop 15 repressors:")
print(top_rep[["motif", "repressor_sum"]].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 8))

ax = axes[0]
tab = top_abs.set_index("motif").iloc[::-1]
ax.barh(tab.index, tab["activator_sum"], color="#2E7D32", label="Activator sum")
ax.barh(tab.index, tab["repressor_sum"], color="#C62828", label="Repressor sum")
ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel("Summed Jacobian*input contribution across neuron genes")
ax.set_title("Top 25 motifs by |contribution| (neurons, FT model)")
ax.legend(loc="lower right")

ax = axes[1]
act = top_act.iloc[::-1]
rep = top_rep.iloc[::-1]
y_act = np.arange(len(act))
y_rep = np.arange(len(rep)) + len(act) + 1
ax.hlines(y_act, 0, act["activator_sum"], color="#2E7D32", lw=2)
ax.plot(act["activator_sum"], y_act, "o", color="#2E7D32")
ax.hlines(y_rep, 0, rep["repressor_sum"], color="#C62828", lw=2)
ax.plot(rep["repressor_sum"], y_rep, "o", color="#C62828")
ax.set_yticks(list(y_act) + list(y_rep))
ax.set_yticklabels(list(act["motif"]) + list(rep["motif"]))
ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel("Signed contribution sum")
ax.set_title("Top 15 activators (green) / 15 repressors (red)")

fig.suptitle("Neuron TF fingerprint (fine-tuned GET)", fontsize=14)
fig.tight_layout()
fig.savefig(OUT_DIR / "fingerprint_neurons.png", dpi=150, bbox_inches="tight")
plt.show()

## Analysis 3: Motif importance vs TF mRNA expression

Resolve each motif-cluster name to its TF gene members via `NrMotifV1`, look up each TF's neuron mRNA (from the preprocessed RNA CSV, `log10(counts/total*1e6+1)`), and take the max TF expression per cluster. Then plot `log1p(max_TF_expr)` vs `log1p(|importance|)` and report Spearman correlation.

In [ ]:
from gcell.dna.nr_motif_v1 import NrMotifV1

nr = NrMotifV1.load_from_pickle()
cluster_to_tfs = nr.get_motifcluster_list_genes()


def resolve_tfs(motif_name):
    if motif_name in cluster_to_tfs:
        return list(cluster_to_tfs[motif_name])
    base = re.sub(r"/\d+$", "", motif_name)
    if base in cluster_to_tfs:
        return list(cluster_to_tfs[base])
    return [motif_name]


rna = pd.read_csv(RNA_CSV)
rna["gene_name"] = rna["gene_name"].astype(str).str.upper()
rna_map = dict(zip(rna["gene_name"], rna["TPM"]))

rows = []
for m in MOTIF_NAMES:
    tfs = [t.upper() for t in resolve_tfs(m)]
    expr_vals = [rna_map.get(t, 0.0) for t in tfs]
    n_expressed = sum(1 for v in expr_vals if v > 0)
    max_expr = max(expr_vals) if expr_vals else 0.0
    rows.append({
        "motif": m,
        "tf_members": ";".join(tfs),
        "n_tfs": len(tfs),
        "n_expressed_tfs": n_expressed,
        "max_tf_expr": max_expr,
        "abs_importance": abs_sum.loc[m],
    })
motif_tf = pd.DataFrame(rows)
motif_tf["resolved"] = motif_tf["tf_members"].apply(lambda s: any(t in rna_map for t in s.split(";")))
motif_tf.to_csv(OUT_DIR / "motif_tf.csv", index=False)

n_unresolved = (~motif_tf["resolved"]).sum()
print("Motif clusters with no TF member present in RNA table:", n_unresolved)
print(motif_tf[~motif_tf["resolved"]][["motif", "tf_members", "abs_importance"]].head(10))

In [ ]:
plot_df = motif_tf[motif_tf["resolved"]].copy()
plot_df["log_expr"] = np.log1p(plot_df["max_tf_expr"])
plot_df["log_imp"] = np.log1p(plot_df["abs_importance"])

rho, p = spearmanr(plot_df["log_expr"], plot_df["log_imp"])
pr, pp = pearsonr(plot_df["log_expr"], plot_df["log_imp"])

top20 = plot_df.sort_values("abs_importance", ascending=False).head(20)
top25_motifs = plot_df.sort_values("abs_importance", ascending=False).head(25)
interp_rate = (top25_motifs["n_expressed_tfs"] > 0).mean()

important_not_expressed = top25_motifs[top25_motifs["max_tf_expr"] == 0][["motif", "tf_members", "abs_importance"]]

fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(
    plot_df["log_expr"], plot_df["log_imp"],
    c=plot_df["n_expressed_tfs"].clip(upper=5),
    cmap="viridis", s=25, alpha=0.8, edgecolor="none",
)
for _, row in top20.iterrows():
    ax.annotate(row["motif"], (row["log_expr"], row["log_imp"]),
                fontsize=8, xytext=(3, 3), textcoords="offset points")
cb = plt.colorbar(sc, ax=ax)
cb.set_label("# TF members with RNA > 0 (clipped at 5)")
ax.set_xlabel("log1p(max TF mRNA in neurons)   [log10-TPM scale]")
ax.set_ylabel("log1p(|sum J*x| over neuron genes)")
ax.set_title(f"Motif importance vs TF mRNA expression (Spearman rho={rho:.3f}, Pearson r={pr:.3f})")
fig.tight_layout()
fig.savefig(OUT_DIR / "motif_vs_tfexpr.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Interpretability rate (top-25 motifs whose TFs have RNA>0): {interp_rate:.2%}")
print(f"Spearman rho = {rho:.3f} (p={p:.2e})")
print(f"Pearson  r   = {pr:.3f} (p={pp:.2e})")
print("\nImportant-but-unexpressed (top-25, max_tf_expr == 0):")
print(important_not_expressed.to_string(index=False))

## Analysis 8: Base vs fine-tuned Jacobian diff

Compare the per-motif absolute importance between the base pretrained model and the LoRA fine-tuned model. Largest positive delta = motifs the fine-tune *upweighted*; largest negative delta = motifs it *downweighted*. Also shows per-gene motif redistribution for neuron-associated marker genes.

In [ ]:
imp_base = mot_base.abs().sum(axis=0)
imp_ft   = mot_ft.abs().sum(axis=0)
delta    = imp_ft - imp_base

diff = pd.DataFrame({
    "motif": MOTIF_NAMES,
    "imp_base": imp_base.values,
    "imp_ft":   imp_ft.values,
    "delta":    delta.values,
    "rel_change": (imp_ft.values - imp_base.values) / (imp_base.values + 1e-12),
})
diff.to_csv(OUT_DIR / "base_vs_ft_motif.csv", index=False)

n_pos_delta = int((diff["delta"] > 0).sum())
print(f"motifs with positive delta (FT > base): {n_pos_delta} / {len(diff)}")
print(f"median delta: {diff['delta'].median():.4f}, median rel_change: {diff['rel_change'].median():.3f}")

top_gain = diff.sort_values("rel_change", ascending=False).head(15).reset_index(drop=True)
top_loss = diff.sort_values("delta",      ascending=True ).head(15).reset_index(drop=True)

print("\nTop 15 gainers (largest relative change, FT/base ratio):")
print(top_gain[["motif", "imp_base", "imp_ft", "delta", "rel_change"]].to_string(index=False))
print("\nTop 15 losers (largest absolute magnitude drop):")
print(top_loss[["motif", "imp_base", "imp_ft", "delta", "rel_change"]].to_string(index=False))

In [ ]:
top_movers = pd.concat([top_gain, top_loss]).drop_duplicates("motif")

fig, ax = plt.subplots(figsize=(9, 9))
is_mover = diff["motif"].isin(top_movers["motif"])
ax.scatter(diff.loc[~is_mover, "imp_base"], diff.loc[~is_mover, "imp_ft"],
           s=14, color="#BDBDBD", alpha=0.6, label="other motifs")
colors = np.where(top_movers["delta"] > 0, "#2E7D32", "#C62828")
ax.scatter(top_movers["imp_base"], top_movers["imp_ft"],
           s=50, c=colors, edgecolor="k", lw=0.4, label="top 30 movers")
for _, row in top_movers.iterrows():
    ax.annotate(row["motif"], (row["imp_base"], row["imp_ft"]),
                fontsize=8, xytext=(3, 3), textcoords="offset points")
lim_min = max(diff[["imp_base", "imp_ft"]].min().min(), 1e-4)
lim_max = diff[["imp_base", "imp_ft"]].max().max() * 1.2
ax.plot([lim_min, lim_max], [lim_min, lim_max], "k--", lw=0.7)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(lim_min, lim_max); ax.set_ylim(lim_min, lim_max)
ax.set_xlabel("base pretrained |contribution| sum")
ax.set_ylabel("LoRA fine-tuned |contribution| sum")
ax.set_title("Per-motif importance: base vs fine-tuned (held-out neurons)")
ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig(OUT_DIR / "base_vs_ft_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 8))

ax = axes[0]
g = top_gain.iloc[::-1]
y = np.arange(len(g))
w = 0.4
ax.barh(y - w / 2, g["imp_base"], w, color="#9E9E9E", label="base")
ax.barh(y + w / 2, g["imp_ft"],   w, color="#2E7D32", label="fine-tuned")
for yi, rc in zip(y, g["rel_change"]):
    ax.annotate(f"{rc*100:+.0f}%", (max(g.iloc[len(g)-1-yi]["imp_base"], g.iloc[len(g)-1-yi]["imp_ft"]), yi),
                fontsize=8, xytext=(3, 0), textcoords="offset points", va="center")
ax.set_yticks(y); ax.set_yticklabels(g["motif"])
ax.set_title("Top 15 gainers by relative change FT/base")
ax.set_xlabel("|contribution| sum")
ax.legend(loc="lower right")

ax = axes[1]
l = top_loss.iloc[::-1]
y = np.arange(len(l))
ax.barh(y - w / 2, l["imp_base"], w, color="#9E9E9E", label="base")
ax.barh(y + w / 2, l["imp_ft"],   w, color="#C62828", label="fine-tuned")
for yi, delta_val in zip(y, l["delta"]):
    ax.annotate(f"{delta_val:+.2f}", (max(l.iloc[len(l)-1-yi]["imp_base"], l.iloc[len(l)-1-yi]["imp_ft"]), yi),
                fontsize=8, xytext=(3, 0), textcoords="offset points", va="center")
ax.set_yticks(y); ax.set_yticklabels(l["motif"])
ax.set_title("Top 15 losers by absolute magnitude drop")
ax.set_xlabel("|contribution| sum")
ax.legend(loc="lower right")

fig.suptitle("Base vs fine-tuned: top movers\n(fine-tune universally shrinks motif magnitudes; gainers = smallest relative loss or actual increase)", fontsize=12)
fig.tight_layout()
fig.savefig(OUT_DIR / "base_vs_ft_topmovers.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
LANDMARK_GENES = ["SOX2", "PAX6", "NES", "TUBB3", "MAP2", "DCX", "NEUROD1", "ASCL1"]
present = [g for g in LANDMARK_GENES if g in gbm_ft.index]
missing = [g for g in LANDMARK_GENES if g not in gbm_ft.index]
print("Landmark genes present in gene-by-motif table:", present)
print("Landmark genes missing (no peak-overlapping TSS in this held-out neuron sample):", missing)

fig, axes = plt.subplots(len(present), 1, figsize=(13, 3 * len(present)))
if len(present) == 1:
    axes = [axes]
for ax, gene in zip(axes, present):
    ft_vec = mot_ft.loc[gene]
    base_vec = mot_base.loc[gene]
    top_idx = ft_vec.abs().sort_values(ascending=False).head(15).index
    merged = pd.DataFrame({
        "motif": top_idx,
        "base": base_vec.loc[top_idx].values,
        "ft":   ft_vec.loc[top_idx].values,
    })
    y = np.arange(len(merged))
    w = 0.4
    ax.barh(y - w / 2, merged["base"], w, color="#9E9E9E", label="base")
    ax.barh(y + w / 2, merged["ft"],   w, color="#1565C0", label="fine-tuned")
    ax.set_yticks(y); ax.set_yticklabels(merged["motif"])
    ax.axvline(0, color="k", lw=0.5)
    ax.set_title(f"{gene}: top 15 motifs by |FT contribution|")
    ax.set_xlabel("Mean J*x across 200 regions (per TSS, averaged)")
    ax.legend(loc="lower right")
    ax.invert_yaxis()
fig.suptitle("Neuron marker gene redistribution (base vs fine-tuned)", fontsize=14)
fig.tight_layout()
fig.savefig(OUT_DIR / "per_gene_shift.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary JSON

Compact machine-readable summary for the run report.

In [ ]:
summary = {
    "n_unique_genes": int(gbm_ft.shape[0]),
    "n_motifs": int(len(MOTIF_NAMES)),
    "analysis_1_top10_activators": top_act.head(10)[["motif", "activator_sum"]].to_dict(orient="records"),
    "analysis_1_top10_repressors": top_rep.head(10)[["motif", "repressor_sum"]].to_dict(orient="records"),
    "analysis_3_spearman_rho": float(rho),
    "analysis_3_spearman_pvalue": float(p),
    "analysis_3_pearson_r": float(pr),
    "analysis_3_interpretability_rate_top25": float(interp_rate),
    "analysis_3_important_but_unexpressed": important_not_expressed.to_dict(orient="records"),
    "analysis_8_motifs_with_positive_delta": n_pos_delta,
    "analysis_8_median_rel_change": float(diff["rel_change"].median()),
    "analysis_8_top5_gainers_by_rel_change": top_gain.head(5)[["motif", "imp_base", "imp_ft", "delta", "rel_change"]].to_dict(orient="records"),
    "analysis_8_top5_losers_by_delta":       top_loss.head(5)[["motif", "imp_base", "imp_ft", "delta", "rel_change"]].to_dict(orient="records"),
    "landmark_genes_plotted": present,
    "landmark_genes_missing": missing,
}
with open(OUT_DIR / "summary_jacobian.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2)[:3000])